In [1]:
%run ./notebook_setup.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from src.utils.config import load_config

/home/inventor/Churn_Prediction_Project


In [2]:
config = load_config()

In [3]:
config

{'paths': {'train_raw_data': '/home/inventor/Churn_Prediction_Project/data/raw/train.csv',
  'test_raw_data': '/home/inventor/Churn_Prediction_Project/data/raw/test.csv',
  'description_of_data': '/home/inventor/Churn_Prediction_Project/data/raw/data_descriptions.csv',
  'preprocessed_data_dir': '/home/inventor/Churn_Prediction_Project/data/preprocessed',
  'model_dir': '/home/inventor/Churn_Prediction_Project/models',
  'output_dir': '/home/inventor/Churn_Prediction_Project/output',
  'X_train_preprocessed': '/home/inventor/Churn_Prediction_Project/data/preprocessed/X_train.npy',
  'X_test_preprocessed': '/home/inventor/Churn_Prediction_Project/data/preprocessed/X_test.npy',
  'X_val_preprocessed': '/home/inventor/Churn_Prediction_Project/data/preprocessed/X_val.npy',
  'y_train_preprocessed': '/home/inventor/Churn_Prediction_Project/data/preprocessed/y_train.npy',
  'y_test_preprocessed': '/home/inventor/Churn_Prediction_Project/data/preprocessed/y_test.npy',
  'y_val_preprocessed': 

In [4]:
# Load the preprocessed data
X_train_path = config["paths"]["X_train_preprocessed"]
X_test_path = config["paths"]["X_test_preprocessed"]
X_val_path = config["paths"]["X_val_preprocessed"]
y_train_path = config["paths"]["y_train_preprocessed"]
y_val_path = config["paths"]["y_val_preprocessed"]

In [5]:
import joblib

X_train = joblib.load(X_train_path)
X_test = joblib.load(X_test_path)
X_val = joblib.load(X_val_path)
y_train = joblib.load(y_train_path)
y_val = joblib.load(y_val_path)

In [6]:
# Train Model on validation dataset
from sklearn.linear_model import LogisticRegression

logr_model = LogisticRegression()

logr_model.fit(X_train, y_train)

y_predict = logr_model.predict(X_val)

In [7]:
from sklearn.metrics import accuracy_score

accuracy_logr = accuracy_score(y_val, y_predict)
print(f"Accuracy of Logistic Regression Model: {accuracy_logr:.3f}%")

Accuracy of Logistic Regression Model: 0.820%


In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

In [9]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=config["data"]["random_state"]),
    # "RandomForestClassifier": RandomForestClassifier(random_state=config["data"]["random_state"]),
    "AdaBoost": AdaBoostClassifier(random_state=config["data"]["random_state"]),
    # "GradientBoost": GradientBoostingClassifier(random_state=config["data"]["random_state"])
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():

    print(f"Evaluating .... {name}")

    roc_scores = cross_val_score(model, X_val, y_val, cv=cv, scoring="roc_auc", n_jobs=-1)
    f1_scores = cross_val_score(model, X_val, y_val, cv=cv, scoring="f1", n_jobs=-1)
    accuracy_scores = cross_val_score(model, X_val, y_val, cv=cv, scoring="accuracy", n_jobs=-1)

    results.append({
        "Model": name,
        "ROC_AUC_Mean":roc_scores.mean(),
        "ROC_AUC_STD":roc_scores.std(),
        "F1 Score Mean":f1_scores.mean(),
        "F1 Score STD": f1_scores.std(),
        "Accuracy Mean": accuracy_scores.mean(),
        "Accuracy Score STD": accuracy_scores.std()
    })

Evaluating .... LogisticRegression


Evaluating .... AdaBoost


,Model,ROC_AUC_Mean,ROC_AUC_STD,F1 Score Mean,F1 Score STD,Accuracy Mean,Accuracy Score STD
0,LogisticRegression,0.752837,0.004536,0.203207,0.008493,0.824931,0.000638
1,AdaBoost,0.743032,0.005181,0.200141,0.009082,0.822737,0.000807


In [11]:
# Comparision Models
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by="ROC_AUC_Mean", ascending=False).reset_index(drop=True)
display(result_df)

,Model,ROC_AUC_Mean,ROC_AUC_STD,F1 Score Mean,F1 Score STD,Accuracy Mean,Accuracy Score STD
0,LogisticRegression,0.752837,0.004536,0.203207,0.008493,0.824931,0.000638
1,AdaBoost,0.743032,0.005181,0.200141,0.009082,0.822737,0.000807


In [12]:
X_train.shape

(195029, 195067)